This notebook provides a tutorial for running the reduction routines provided with LIRAS.

First, let's import the functions we need from the routines. Make sure the routines are in the same folder as this notebook.

In [ ]:
from Dark import dark
from Flat import flat
from Reduce import reduce_noflats, reduce_withflats
from DualSplit import dualsplit
from SkySub import skysub
from Rotate import rot
from AlignandStack import alignandstack

### Dark.py
The first step is to create the median dark and bad pixel map that will be used to calibrate the raw science files. You'll need a folder of darks taken on the same night as your science observations.

The inputs you will need for the function are:
1) the path to the dark files
2) defining whether the image is a cube or a single image (cube == True or cube == False)
3) define whether you want to median an image cube if there is one (median == True or median == False)
4) define which frame in the cube you want to use if not taking the median of the cube

In [ ]:
path = '/Users/marah/Downloads/files/darks/1.5s/'

cube = True ###is the image a cube?

median = False ###do you want to median the cube?

frame = 1 ###frame in cube you want to use

dark(path,cube,median,frame)

If you have science images in the J, H, or K near-infrared bands, flat observations are also required for calibration. You'll need to get a median dark for the flat files as well using the Dark.py routine.
You'll need the same inputs as above:
1) the path to the darks for flats files
2) defining whether the image is a cube or a single image (cube == True or cube == False)
3) define whether you want to median an image cube if there is one (median == True or median == False)
4) define which frame in the cube you want to use if not taking the median of the cube

In [ ]:
path = '/Users/marah/Downloads/files/flats/darksforflats/'

cube = True ###is the image a cube?

median = False ###do you want to median the cube?

frame = 1 ###frame in cube you want to use

dark(path,cube,median,frame)

### Flat.py
If you have science images in the J, H, or K near-infrared bands, flat observations are also required for calibration. The next step is to create the median flat and a flat bad pixel map that will be used to calibrate the raw science files. You'll need a folder of flats taken on the same night as your science observations.

The inputs you will need for the function are:
1) the path to the flat files
2) the path to the median dark for flats created in the previous step

In [ ]:
flatpath = '/Users/marah/Downloads/files/flats/K/'
flatdarkpath = '/Users/marah/Downloads/files/flats/darksforflats/reduced/median_dark_cln.fits'


flat(flatpath,flatdarkpath)

### Reduce.py
With the necessary calibration files in hand, we can now perform the standard astronomical data reduction procedures on the raw data. If your data is in the L- or M-bands, the inputs you will need for the function are:

1) the path to the median dark file
2) the path to the bad pixel map created using the darks
3) the path to the raw science frames

In [ ]:
darkpath = '/Users/marah/Downloads/files/darks/1.5s/reduced/median_dark_cln.fits'
bpath = '/Users/marah/Downloads/files/darks/1.5s/reduced/bpmask_dark.fits'
sciencepath = '/Users/marah/Downloads/files/science/'

reduce_noflats(darkpath,bpath,sciencepath)

With the necessary calibration files in hand, we can now perform the standard astronomical data reduction procedures on the raw data. If your data is in the J-, H-, or K-bands, the inputs you will need for the function are:

1) the path to the median dark file
2) the path to the median flat file
3) the path to the bad pixel map created using the darks
4) the path to the raw science frames

In [ ]:
darkpath = '/Users/marah/Downloads/files/darks/1.5s/reduced/median_dark_cln.fits'
flatpath = '/Users/marah/Downloads/files/flats/K/reduced/median_flat_cln_scl.fits'
bpath = '/Users/marah/Downloads/files/darks/1.5s/reduced/bpmask_dark.fits'
sciencepath = '/Users/marah/Downloads/files/science/'

reduce_withflats(darkpath,flatpath,bpath,sciencepath)

### DualSplit.py
If your science data was taken with both apertures of LBT (so the SX and DX sides = dual aperture data) it is required to separate the observations from the two apertures, since the distortion and orientation solutions are slightly different between the two. The input you will need for this function is:

1) the path to the reduced science files

In [ ]:
pathname = '/Users/marah/Downloads/files/science/reduced/'

dualsplit(pathname)

### SkySub.py
The next step is to remove the signal of the background in the observations using sky subtraction. If you separated the data using the DualSplit.py routine, make sure to perform sky subtraction on both the SX and DX data. The inputs you will need are:

1) the path to the reduced science files
2) the filename for a sky background file that will be generated
3) the "file midpoint" number that determines when the A nods turn into B nods. Example: if you have 600 total files, usually the first 300 will be A nods and the last 300 will be B nods, so the file midpoint would be 300. 

In [ ]:
pathname = '/Users/marah/Downloads/files/science/reduced/DX/'
skypathname = '/Users/marah/Downloads/files/science/reduced/DX/sky/SkyBG.fits'
filemidpoint = 10

skysub(pathname,skypathname,filemidpoint)

### Rotate.py
The next step is to rotate each file to be oriented to true North. If you separated the data using the DualSplit.py routine, make sure to rotate both the SX and DX data separately. The inputs you will need are:

1) the LMIRCam detector offset either given by the LMIRCam team or determined using the dewarp package and observations of astrometric fields
2) the path to the reduced and sky-subtracted science files
3) the path to save the rotated files


In [ ]:
lm_offset = 1.7
pathname = '/Users/marah/Downloads/files/science/reduced/DX/SkySub/'
savepath = '/Users/marah/Downloads/files/science/reduced/DX/SkySub/rotated/'

rot(lm_offset,pathname,savepath)

In [ ]:
lm_offset = -0.7
pathname = '/Users/marah/Downloads/files/science/reduced/SX/SkySub/'
savepath = '/Users/marah/Downloads/files/science/reduced/SX/SkySub/rotated/'

rot(lm_offset,pathname,savepath)

### AlignandStack.py
The next step is to align the target observed in each file to the same central pixel, and then stack the images through median combination. If you separated the data using the DualSplit.py routine, make sure to align and stack both the SX and DX data separately. The inputs you will need are:

1) the path to the reduced, sky-subtracted, and rotated science files
2) the path where the aligned files will be saved
3) the name of the final fits image after median combination
4) the threshold value of the background in the image (used for finding the brightest pixel in the image which gives the star coordinates)


In [ ]:
pathforalign = '/Users/marah/Downloads/files/science/reduced/DX/SkySub/rotated/'
pathforstack = '/Users/marah/Downloads/files/science/reduced/DX/SkySub/rotated/align/'

finalimage = 'V927Tau_K.fits'

thresholdvalue = 15

alignandstack(pathforalign,pathforstack,finalimage,thresholdvalue)